# Learning to use NLP features and Pipelines

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#preprocessing
from sklearn.preprocessing import StandardScaler, Normalizer, MinMaxScaler

# pipelines
from sklearn.pipeline import Pipeline

# NLP transformers
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

# classifiers you can use
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# model selection bits
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score, ShuffleSplit
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold, KFold

# evaluation
from sklearn.metrics import f1_score, accuracy_score

# dataset
from sklearn.datasets import fetch_20newsgroups

# plotting
from plotting import plot_learning_curve, plot_validation_curve

# Loading the data

In [ ]:
data = fetch_20newsgroups(categories = [ 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.religion.misc'],remove=('headers', 'footers', 'quotes'))

In [ ]:
data.keys()

In [ ]:
for i,name in enumerate(data.target_names):
    print (f"{i} is {name}")

In [ ]:


# Only relabel if there are more than two unique labels (i.e., not just [0, 1])
if set(np.unique(data.target)) != {0, 1}:
    # relabel targets using a mapping dict for clarity and scalability
    label_map = {0: 0, # computer
                 1: 0, # computer
                 2: 0, # computer
                 3: 1, # talk
                 4: 1, # talk
                 5: 1} # talk
    data.target = np.array([label_map[t] for t in data.target])

In [ ]:
data.target_names = ["computer",'talk']

In [ ]:
# checking that I only have two classes left!
np.unique(data.target)

In [ ]:
# let's check the distribution of my two classes
np.bincount(data.target)

In [ ]:
X = data.data
X[0]

In [ ]:
print(X[0])

In [ ]:
data.target_names

In [ ]:
y = data.target
data.target_names[y[0]]

In [ ]:
# split into training / testing
# note I'm setting a random_state, normally I wouldn't but... I want to be able to describe the data to you, and I need to be
# sure you have the same split I get.
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify = y, test_size = 0.2, random_state=20)

### Extract features.
## A toy example first

In [ ]:
toys = ["this is the first sentence in this list", "this is the second sentence"]
test_count = CountVectorizer()
test_count.fit(toys)

In [ ]:
test_count.vocabulary_

In [ ]:
toys_transformed = test_count.transform(toys)
toys_transformed

In [ ]:
# we can transform it into a full array, but you shouldn't really do that....
toys_transformed.toarray()

In [ ]:
print(toys_transformed)

In [ ]:
# count vectorizer, easiest thing to do. Will make n-grams for us.
count_vect  = CountVectorizer()
count_vect.fit(X_train)

In [ ]:
len(count_vect.vocabulary_)

In [ ]:
X_train_vect = count_vect.transform(X_train)

In [ ]:
type(X_train_vect)

In [ ]:
X_train_vect.shape

In [ ]:
print(X_train_vect[0].getnnz())  #getnnz --> "get non zero" will tell us how many non-zero entries we have

In [ ]:
print(X_train_vect[0].sum())

# Next step - train our baseline model

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
scores = cross_val_score(dummy, X_train, y_train, cv=3, scoring = "accuracy") 
print (f"{scores.mean()} is the mean score")
scores

# Training a real model - with pipelines

In [ ]:
# create the pipeline
dt_pipe = Pipeline([('vect', CountVectorizer(ngram_range=(1,1))),('clf', DecisionTreeClassifier())])

# pass the pipeline as if it was the classifier into a cross validation method
# the cv method will automatically figure out what to do.
scores = cross_val_score(dt_pipe, X_train, y_train, cv=5, scoring = 'accuracy')
print(f"the mean score is: {scores.mean()}")
print("The scores were:", scores)

# Getting better features

In [ ]:
# let's make unigrams and bigrams
count = CountVectorizer(ngram_range=(1,2))
clf = DecisionTreeClassifier()
dt_pipe = Pipeline([('vect', count),('clf', clf)])
scores = cross_val_score(dt_pipe, X_train, y_train, cv=5, scoring = 'accuracy')
print(scores.mean())
scores

In [ ]:
len(count.fit(X_train).vocabulary_)

In [ ]:
# let's make unigrams and bigrams, we will limit to 65k features
count = CountVectorizer(ngram_range=(1,2), max_features=65_000)  #yes you can use _'s in integers!

clf = DecisionTreeClassifier()
dt_pipe = Pipeline([('vect', count),('clf', clf)])
scores = cross_val_score(dt_pipe, X_train, y_train, cv=5, scoring = 'accuracy')
print(scores.mean())
scores

In [ ]:
# let's make unigrams , we will limit to 10k features
count = CountVectorizer(ngram_range=(1,1), max_features=10_000)  #yes you can use _'s in integers!
clf = DecisionTreeClassifier()
#tf-idf
tfidf = TfidfTransformer()

dt_pipe = Pipeline([('vect', count),('tfidf',tfidf),('clf', clf)])
scores = cross_val_score(dt_pipe, X_train, y_train, cv=5, scoring = 'accuracy')
print(scores.mean())
scores

In [ ]:
count = CountVectorizer(ngram_range=(1,3), max_features=10_000)  #yes you can use _'s in integers!
clf = DecisionTreeClassifier()
#tf-idf
tfidf = TfidfTransformer()

dt_pipe = Pipeline([('vect', count),('tfidf',tfidf),('clf', clf)])
scores = cross_val_score(dt_pipe, X_train, y_train, cv=5, scoring = 'accuracy')
print(scores.mean())
scores

# Your job.